[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fw-ai/cookbook/blob/main/training/case-studies/sft_prompt_router/prompt_router_serverless.ipynb)

# SFT: prompt-routing classifier — **serverless** training loop

The serverless twin of `sft_prompt_router/`. Same task — fine-tune a small model to be a
**prompt router** that emits a 5-field JSON (`complexity`, `math`, `code`, `reasoning`,
`route`) — but instead of a managed `supervised_fine_tuning_jobs` job plus on-demand
deployments, it runs a **Tinker-style loop against Fireworks serverless training**.

**What changes vs the managed notebook:**

| | Managed (`sft_prompt_router`) | Serverless (this notebook) |
| --- | --- | --- |
| Client | `Fireworks()` REST SDK | `FiretitanServiceClient(base_url=".../training/v1/serverless")` |
| Training | `supervised_fine_tuning_jobs.create(...)` + poll | your own `forward_backward("cross_entropy")` + `optim_step` loop |
| Provisioning | trainer job + on-demand deployment you tear down | none — attach to a shared pooled trainer |
| Sampling / eval | deploy the model, score, delete | in-session `create_sampling_client(snapshot)` — no deployment |
| Data | upload a JSONL dataset resource | render chat rows to `tinker.Datum` client-side |

**Fair comparison (unchanged):** the **base** model is scored with the *rich* prompt (routing
rule spelled out — best-effort prompt engineering); the **tuned** model is scored with the
*lean* prompt (schema only) because it learned the policy during SFT. “Base” here is the
freshly-attached LoRA adapter before any optimizer step (identity → base behavior).

> Serverless is **LoRA only** and requires a supported base model and a concrete
> `max_seq_len`. Training + sampling are billed per token; there is no idle cost and nothing
> to tear down. See `serverless.md`.

## Prerequisites

```bash
pip install --pre "fireworks-ai[training]>=1.2.0a87,<2" python-dotenv datasets  # serverless adapter serving needs >=1.2.0a87
# tinker + tinker_cookbook (renderers/tokenizers) come with the training extra
```

`.env`: `FIREWORKS_API_KEY` (and optionally `FIREWORKS_BASE_URL`).

## 1. Setup & connect to the serverless session

One `FiretitanServiceClient` pointed at the serverless surface gives us **both** a training
client and (later) a sampling client. `create_lora_training_client` attaches to the shared
pooled trainer for `BASE_MODEL` — that attachment is the training session.

In [1]:
import json
import math
import os
import random

import dotenv
import tinker
from fireworks.training.sdk import FiretitanServiceClient
from tinker_cookbook.model_info import get_recommended_renderer_name
from tinker_cookbook.renderers import TrainOnWhat, get_renderer, get_text_content
from tinker_cookbook.tokenizer_utils import get_tokenizer

dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True), override=True)  # repo-root .env
assert os.getenv("FIREWORKS_API_KEY"), "FIREWORKS_API_KEY missing from .env"


def serverless_base_url(base_url: str) -> str:
    """Serverless training + sampling both hang off /training/v1/serverless."""
    root = base_url.rstrip("/")
    if root.endswith("/training/v1/serverless"):
        return root
    if root.endswith("/training/v1"):
        return f"{root}/serverless"
    return f"{root}/training/v1/serverless"


import importlib.metadata as _md
from packaging.version import parse as _vparse
_fw = _md.version("fireworks-ai")
print("imports ready | fireworks-ai", _fw, "| tinker", _md.version("tinker"))
# Serverless adapter serving requires fireworks-ai >= 1.2.0a87 (older builds silently serve base).
assert _vparse(_fw) >= _vparse("1.2.0a87"), "upgrade: pip install --pre 'fireworks-ai[training]>=1.2.0a87,<2'"

/Users/sinan/cookbook-casestudies/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports ready | fireworks-ai 1.2.0 | tinker 0.22.7


## 2. Configuration

The label schema, sample sizes, and loop knobs. `BASE_MODEL` must be a serverless-supported
model and `TOKENIZER_MODEL` must match it (a mismatch corrupts rendering/decoding). Serverless
has no training instance to infer sequence length from, so `MAX_SEQ_LEN` is set explicitly.

In [2]:
# CONFIG
# Serverless-supported base + its matching HF tokenizer (see serverless.md).
BASE_MODEL = "accounts/fireworks/models/qwen3p5-9b"
TOKENIZER_MODEL = "Qwen/Qwen3.5-9B"
# tinker_cookbook's registry has the Qwen3.5 family (renderer "qwen3_5") but no "Qwen3.5-9B"
# entry, so auto-lookup KeyErrors. Use the DISABLE-THINKING variant: this is a /no_think
# single-turn classifier, so it must render + parse WITHOUT a thinking block so the model emits
# the JSON directly and parse_response returns it. The standard "qwen3_5" renderer primes a
# <think> block -> the model burns the token budget reasoning, no JSON is parsed -> ~3% on every
# field (base included). disable_thinking is the client-side equivalent of how the managed
# deployment serves this with /no_think.
RENDERER_NAME = "qwen3_5_disable_thinking"
HF_DATASET = "SupraLabs/Prompt-Routing-Dataset"

# Fields the classifier predicts + allowed values. Features FIRST, route LAST, so the model
# "computes" the decision inputs before committing to the route.
LABELS = {
    "complexity": {"values": [1, 2, 3, 4, 5]},
    "math":       {"values": [True, False]},
    "code":       {"values": [True, False]},
    "reasoning":  {"values": [True, False]},
    "route":      {"values": ["small model", "big model"]},
}
FIELDS = list(LABELS)
PRIMARY_FIELD = "route"  # the actual decision; headline metric

# Sample sizes (dataset has 992 rows; bump for more signal).
N_TRAIN = 600
N_EVAL = 60
SEED = 0

# SFT loop knobs
EPOCHS = 3
BATCH_SIZE = 32
LORA_RANK = 8            # serverless is LoRA-only: rank must be > 0
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = 2048       # required on serverless (no training shape to infer it from)
MAX_EVAL_TOKENS = 512

# Serverless per-token pricing for BASE_MODEL (serverless.md: qwen3p5-9b, 64K). Single-turn
# eval doesn't reuse KV cache, so the full prefill rate applies (no cached-prefill discount).
PREFILL_PER_1M = 0.44
SAMPLE_PER_1M = 1.33

API_BASE_URL = os.environ.get("FIREWORKS_BASE_URL", "https://api.fireworks.ai")
print("base_model:", BASE_MODEL, "| tokenizer:", TOKENIZER_MODEL)

base_model: accounts/fireworks/models/qwen3p5-9b | tokenizer: Qwen/Qwen3.5-9B


In [ ]:
# Tokenizer + renderer (client-side: render prompts, decode sampled tokens).
tokenizer = get_tokenizer(TOKENIZER_MODEL)
renderer_name = RENDERER_NAME or get_recommended_renderer_name(TOKENIZER_MODEL)
renderer = get_renderer(renderer_name, tokenizer)

# The one connection that hands back BOTH training and sampling clients. No trainer job,
# no deployment -- just a pooled serverless session.
service = FiretitanServiceClient(
    api_key=os.environ["FIREWORKS_API_KEY"],
    base_url=serverless_base_url(API_BASE_URL),
)
training_client = service.create_lora_training_client(base_model=BASE_MODEL, rank=LORA_RANK)

session = getattr(service, "training_session_id", None)
print(f"connected: session={session} run={getattr(training_client, 'run_id', None)}")
print(f"renderer={renderer_name}")
assert session, "no training_session_id -- is base_url pointed at /training/v1/serverless?"

connected: session=ts-a35c0e03d82a4c498d7732b6099c9223 run=run-1f4951fc31b540fcaf5866b25d753aaa
renderer=qwen3_5_disable_thinking


## 3. Build the dataset (rendered to `tinker.Datum`)

Download the routing dataset, define the two system prompts (`LEAN_PROMPT` for training /
tuned eval, `RICH_PROMPT` for the base eval), and render each training row to a cross-entropy
`tinker.Datum`. On the serverless path there is no dataset resource to upload: the renderer
turns `[system, user, assistant]` into tokens client-side, masks everything but the
assistant JSON (via `train_on_what`), and we hand the shifted next-token layout straight to
`forward_backward`.

In [4]:
from datasets import load_dataset


def _fmt_values(vals):
    return ", ".join(json.dumps(v) for v in vals)


# LEAN = schema only (train + tuned eval). RICH = LEAN + the routing rule (base eval).
_SCHEMA = (
    "Respond with ONLY a compact JSON object (no prose, no code fence) with exactly these "
    "keys, in this order:\n"
    f"- \"complexity\": integer {_fmt_values(LABELS['complexity']['values'])} (1=trivial, 5=advanced)\n"
    f"- \"math\": {_fmt_values(LABELS['math']['values'])} (needs symbolic math / proofs / multi-step word math)\n"
    f"- \"code\": {_fmt_values(LABELS['code']['values'])} (needs code generation / debugging)\n"
    f"- \"reasoning\": {_fmt_values(LABELS['reasoning']['values'])} (needs multi-step reasoning)\n"
    f"- \"route\": one of {_fmt_values(LABELS['route']['values'])}\n\n"
    'Example: {"complexity": 3, "math": true, "code": false, "reasoning": true, "route": "big model"}'
)
_INTRO = (
    "You are a prompt-routing classifier for an LLM serving stack. Given a user prompt, "
    "analyze the task and decide whether a small local model can handle it or whether it "
    "must be escalated to a big frontier model.\n\n"
)
_RULE = (
    "Routing policy: choose \"big model\" if complexity >= 3 OR the task needs code OR the "
    "task needs math; otherwise choose \"small model\".\n\n"
)
_NOTHINK = "\n/no_think"  # Qwen hybrid-thinking: emit the JSON directly, no reasoning channel

LEAN_PROMPT = _INTRO + _SCHEMA + _NOTHINK
RICH_PROMPT = _INTRO + _RULE + _SCHEMA + _NOTHINK


def gold_label(ex) -> dict:
    return {
        "complexity": int(ex["complexity_score"]),
        "math": bool(ex["math_task"]),
        "code": bool(ex["coding_task"]),
        "reasoning": bool(ex["requires_reasoning"]),
        "route": ex["routing_choice"],
    }


def train_messages(ex) -> list[dict]:
    return [
        {"role": "system", "content": LEAN_PROMPT},
        {"role": "user", "content": ex["prompt"]},
        {"role": "assistant", "content": json.dumps(gold_label(ex))},
    ]


ds = load_dataset(HF_DATASET, split="train").shuffle(seed=SEED)
train_ex = [ds[i] for i in range(N_TRAIN)]
eval_ex = [ds[i] for i in range(N_TRAIN, N_TRAIN + N_EVAL)]
print(f"{len(train_ex)} train rows, {len(eval_ex)} eval rows")

600 train rows, 60 eval rows


In [5]:
# Render chat messages -> cross-entropy tinker.Datum (shifted next-token layout).
def _to_ints(rendered_input):
    if isinstance(rendered_input, tinker.ModelInput):
        return [int(x) for x in rendered_input.to_ints()]
    return [int(x) for x in (rendered_input.tolist() if hasattr(rendered_input, "tolist") else rendered_input)]


def build_sft_datum(messages, max_seq_len=MAX_SEQ_LEN):
    """Render [system, user, assistant] to a cross_entropy datum, or None if too short.

    The renderer returns full-sequence tokens + per-token weights (1.0 on the assistant
    JSON, 0.0 elsewhere). We shift to next-token prediction: the model sees tokens[:-1],
    targets are tokens[1:], and weights[1:] select the response tokens the loss trains on.
    """
    rendered_input, weights = renderer.build_supervised_example(
        messages, train_on_what=TrainOnWhat.LAST_ASSISTANT_TURN
    )
    tokens = _to_ints(rendered_input)
    w = [float(x) for x in (weights.tolist() if hasattr(weights, "tolist") else weights)]
    tokens, w = tokens[:max_seq_len], w[:max_seq_len]
    if len(tokens) < 2 or not any(x > 0 for x in w[1:]):
        return None  # nothing to learn from (empty target / truncated away)
    return tinker.Datum(
        model_input=tinker.ModelInput.from_ints(tokens[:-1]),
        loss_fn_inputs={
            "target_tokens": [int(t) for t in tokens[1:]],
            "weights": [float(x) for x in w[1:]],
        },
    )


train_datums = [d for ex in train_ex if (d := build_sft_datum(train_messages(ex))) is not None]
print(f"rendered {len(train_datums)}/{len(train_ex)} training datums")

# Mirror the managed job's eval_auto_carveout=True: hold out 10% (seeded, capped at 100) of the
# training rows so we train on the same ~540 the dedicated run did (there they become the job's
# internal eval; here we just exclude them from the gradient).
_carve_n = min(int(len(train_datums) * 0.1), 100)
_idx = list(range(len(train_datums)))
random.Random(SEED).shuffle(_idx)
_carved = set(_idx[:_carve_n])
train_datums = [d for i, d in enumerate(train_datums) if i not in _carved]
print(f"training on {len(train_datums)} datums ({_carve_n} carved out to match managed eval_auto_carveout)")
assert train_datums, "no training datums rendered -- check tokenizer/renderer"

rendered 600/600 training datums
training on 540 datums (60 carved out to match managed eval_auto_carveout)


## 4. Evaluation helpers (in-session sampling)

No deployment: we `save_weights_for_sampler(...)` to snapshot the current adapter, open a
sampling client bound to that exact snapshot, and score the held-out rows. The reward parses
the JSON and scores it **per field** against the gold label.

In [6]:
def parse_pred(text: str) -> dict:
    """Pull the JSON object out of the model's answer (tolerates prose / code fences)."""
    if not text:
        return {}
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return {}
    try:
        obj = json.loads(text[start:end + 1])
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}


def _norm(v):
    return str(v).strip().lower()


def _predict(sampler, messages):
    prompt = renderer.build_generation_prompt(messages)
    params = tinker.SamplingParams(
        max_tokens=MAX_EVAL_TOKENS, temperature=0.0, stop=renderer.get_stop_sequences()
    )
    res = sampler.sample(prompt=prompt, num_samples=1, sampling_params=params)
    res = res.result() if hasattr(res, "result") else res
    seq = res.sequences[0]
    return get_text_content(renderer.parse_response(list(seq.tokens))[0])


def eval_snapshot(snapshot, examples, label, system_prompt):
    """Score `examples` through a sampler bound to `snapshot`. Returns (exact_match, per_field)."""
    sampler = service.create_sampling_client(model_path=snapshot, tokenizer=tokenizer)
    per_totals = {f: 0 for f in FIELDS}
    exact = 0
    try:
        for ex in examples:
            msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": ex["prompt"]}]
            pred = parse_pred(_predict(sampler, msgs))
            gold = gold_label(ex)
            per = {f: int(f in pred and _norm(pred[f]) == _norm(gold[f])) for f in FIELDS}
            for f in FIELDS:
                per_totals[f] += per[f]
            exact += int(all(per[f] for f in FIELDS))
    finally:
        sampler.close()
    n = len(examples)
    per_acc = {f: per_totals[f] / n for f in FIELDS}
    print(f"[{label}] exact-match {exact}/{n} = {exact / n:.1%} | route {per_acc[PRIMARY_FIELD]:.1%}")
    return exact / n, per_acc


print("eval helpers ready")

eval helpers ready


## 5. Evaluate the base model (before)

Snapshot the freshly-attached adapter (no optimizer step yet → base behavior) and score it
with the **rich prompt** — the base model's best-effort setup. This is the bar the fine-tune
has to clear with only the lean prompt.

In [7]:
base_snapshot = training_client.save_weights_for_sampler("router-base").result().path
print("base snapshot:", base_snapshot)
base_exact, base_fields = eval_snapshot(base_snapshot, eval_ex, "base-rich", RICH_PROMPT)

base snapshot: pyroworks/run-1f4951fc31b540fcaf5866b25d753aaa/router-base-1ca6ce72
[base-rich] exact-match 16/60 = 26.7% | route 90.0%


## 6. Train the router (GO LIVE)

The SFT loop: for each epoch, shuffle the datums, and for each batch run one
`forward_backward("cross_entropy")` + `optim_step`. Every call returns a future, so we always
`.result()` it. Loss should fall as the model learns to emit the routing JSON.

**This runs real GPU work and is billed per token.**

In [8]:
def _mean_loss(fb_output):
    """Mean NLL from a forward_backward result: loss:sum / response_tokens."""
    metrics = getattr(fb_output, "metrics", None) or {}
    loss_sum = metrics.get("loss:sum")
    tokens = metrics.get("response_tokens") or metrics.get("num_loss_tokens") or 1.0
    return None if loss_sum is None else float(loss_sum) / max(float(tokens), 1.0)


# Verified against the managed job (GET .../supervisedFineTuningJobs/l2azpnow):
#   learningRate=1e-4, learningRateWarmupSteps=0, lrScheduler=null  -> constant LR, no warmup
#   optimizerWeightDecay=0                                          -> weight_decay=0.0
# beta1/beta2 and grad clipping aren't exposed by the job API, so those follow standard
# AdamW-for-SFT (beta2=0.999) + light grad clipping. (The earlier bad run used the RL-style
# beta2=0.95 + alpha=32, which overfit: train loss -> ~0.01, route collapsed.)
adam = tinker.AdamParams(
    learning_rate=LEARNING_RATE, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0, grad_clip_norm=1.0
)
rng = random.Random(SEED)
step = 0
for epoch in range(EPOCHS):
    order = list(range(len(train_datums)))
    rng.shuffle(order)
    for start in range(0, len(order), BATCH_SIZE):
        batch = [train_datums[i] for i in order[start:start + BATCH_SIZE]]
        fb = training_client.forward_backward(batch, "cross_entropy").result()
        training_client.optim_step(adam).result()
        step += 1
        loss = _mean_loss(fb)
        print(f"epoch {epoch + 1}/{EPOCHS} step {step:03d} "
              f"loss={'n/a' if loss is None else f'{loss:.4f}'} (batch={len(batch)})")

print(f"done: {step} optimizer steps")

epoch 1/3 step 001 loss=0.1020 (batch=32)
epoch 1/3 step 002 loss=0.0589 (batch=32)
epoch 1/3 step 003 loss=0.0433 (batch=32)
epoch 1/3 step 004 loss=0.0542 (batch=32)
epoch 1/3 step 005 loss=0.0463 (batch=32)
epoch 1/3 step 006 loss=0.0604 (batch=32)
epoch 1/3 step 007 loss=0.0330 (batch=32)
epoch 1/3 step 008 loss=0.0512 (batch=32)
epoch 1/3 step 009 loss=0.0373 (batch=32)
epoch 1/3 step 010 loss=0.0305 (batch=32)
epoch 1/3 step 011 loss=0.0265 (batch=32)
epoch 1/3 step 012 loss=0.0425 (batch=32)
epoch 1/3 step 013 loss=0.0438 (batch=32)
epoch 1/3 step 014 loss=0.0260 (batch=32)
epoch 1/3 step 015 loss=0.0291 (batch=32)
epoch 1/3 step 016 loss=0.0399 (batch=32)
epoch 1/3 step 017 loss=0.0327 (batch=28)
epoch 2/3 step 018 loss=0.0246 (batch=32)
epoch 2/3 step 019 loss=0.0225 (batch=32)
epoch 2/3 step 020 loss=0.0259 (batch=32)
epoch 2/3 step 021 loss=0.0240 (batch=32)
epoch 2/3 step 022 loss=0.0270 (batch=32)
epoch 2/3 step 023 loss=0.0345 (batch=32)
epoch 2/3 step 024 loss=0.0319 (ba

## 7. Evaluate the tuned model (after) & compare

Snapshot the trained adapter and score it with the **lean prompt** (no routing rule — it
learned the policy). Headline is `route` accuracy; exact-match and per-field are secondary.
Tuned (lean) matching or beating base (rich) means the fine-tune earned its keep.

In [9]:
tuned_snapshot = training_client.save_weights_for_sampler("router-tuned").result().path
print("tuned snapshot:", tuned_snapshot)
tuned_exact, tuned_fields = eval_snapshot(tuned_snapshot, eval_ex, "tuned-lean", LEAN_PROMPT)

rb, rt = base_fields[PRIMARY_FIELD], tuned_fields[PRIMARY_FIELD]
print("\n===== serverless prompt-router: base (rich) vs tuned (lean) =====")
print(f"  ROUTE accuracy (the decision):  {rb:.1%}  ->  {rt:.1%}   ({rt - rb:+.1%})   <- headline")
print(f"  exact-match (all {len(FIELDS)} fields):  {base_exact:.1%}  ->  {tuned_exact:.1%}"
      f"   ({tuned_exact - base_exact:+.1%})")
print("  per-field accuracy:")
print(f"    {'field':<12}{'base':>8}{'tuned':>8}{'delta':>8}")
for f in FIELDS:
    b, t = base_fields[f], tuned_fields[f]
    print(f"    {f:<12}{b:>7.0%}{t:>8.0%}{t - b:>+8.0%}")
print("================================================================")

tuned snapshot: pyroworks/run-1f4951fc31b540fcaf5866b25d753aaa/router-tuned-1ca6ce72
[tuned-lean] exact-match 43/60 = 71.7% | route 96.7%

===== serverless prompt-router: base (rich) vs tuned (lean) =====
  ROUTE accuracy (the decision):  90.0%  ->  96.7%   (+6.7%)   <- headline
  exact-match (all 5 fields):  26.7%  ->  71.7%   (+45.0%)
  per-field accuracy:
    field           base   tuned   delta
    complexity      52%     75%    +23%
    math            97%     97%     +0%
    code            97%    100%     +3%
    reasoning       52%    100%    +48%
    route           90%     97%     +7%


## Speed & cost on the holdout (serverless)

Single-stream serving benchmark of the tuned adapter over the full holdout, matching the
dedicated notebook: wall-clock, per-request latency, token throughput, and cost. Serverless
bills **per token**, so cost is exact (prompt tokens x prefill rate + output tokens x sample
rate) with **no idle/provisioning cost and nothing to tear down**.

In [10]:
# Benchmark the tuned adapter over the full holdout via the in-session sampler, capturing
# per-request latency + token counts. Sequential (single-stream) to match the dedicated run.
import time


def benchmark_serverless(snapshot, examples, system_prompt, label):
    sampler = service.create_sampling_client(model_path=snapshot, tokenizer=tokenizer)
    params = tinker.SamplingParams(max_tokens=MAX_EVAL_TOKENS, temperature=0.0, stop=renderer.get_stop_sequences())
    lats, prompt_tok, out_tok = [], 0, 0
    t0 = time.time()
    try:
        for ex in examples:
            msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": ex["prompt"]}]
            prompt = renderer.build_generation_prompt(msgs)
            t1 = time.time()
            res = sampler.sample(prompt=prompt, num_samples=1, sampling_params=params)
            res = res.result() if hasattr(res, "result") else res
            lats.append(time.time() - t1)
            seq = res.sequences[0]
            prompt_tok += int(prompt.length)          # billed as prefill tokens
            out_tok += len(list(seq.tokens))           # billed as sample tokens
    finally:
        sampler.close()
    wall = time.time() - t0
    n = len(examples)
    token_cost = prompt_tok / 1e6 * PREFILL_PER_1M + out_tok / 1e6 * SAMPLE_PER_1M   # actual serverless billing
    res = {
        "n": n, "wall_s": wall, "req_per_s": n / wall, "avg_latency_s": sum(lats) / n,
        "prompt_tokens": prompt_tok, "output_tokens": out_tok, "out_tok_per_s": out_tok / wall,
        "token_cost_usd": token_cost, "cost_per_1k_req_usd": token_cost / n * 1000,
    }
    print(f"\n============== {label}: speed & cost on {n} holdout rows ==============")
    print(f"  wall-clock        : {wall:.1f}s   ({res['req_per_s']:.2f} req/s, avg {res['avg_latency_s']:.2f}s/req)")
    print(f"  tokens            : {prompt_tok:,} prompt + {out_tok:,} output   ({res['out_tok_per_s']:.0f} out tok/s)")
    print(f"  COST (serverless) : ${token_cost:.4f}  (prefill ${PREFILL_PER_1M}/1M + sample ${SAMPLE_PER_1M}/1M)")
    print(f"                      -> ${res['cost_per_1k_req_usd']:.2f} per 1k requests")
    print("  note: no provisioning or idle cost — you pay only for these tokens.")
    print("=====================================================================")
    return res


serverless_bench = benchmark_serverless(tuned_snapshot, eval_ex, LEAN_PROMPT, "serverless tuned (qwen3p5-9b)")


============== serverless tuned (qwen3p5-9b): speed & cost on 60 holdout rows ==============
  wall-clock        : 31.3s   (1.92 req/s, avg 0.52s/req)
  tokens            : 17,095 prompt + 1,860 output   (59 out tok/s)
  COST (serverless) : $0.0100  (prefill $0.44/1M + sample $1.33/1M)
                      -> $0.17 per 1k requests
  note: no provisioning or idle cost — you pay only for these tokens.


## 8. Save a final checkpoint & clean up

Save the final adapter snapshot and close the session. There is **no deployment to tear
down** on the serverless path — closing the service releases the pooled attachment; billing
is per token, so an idle session costs nothing. To serve this adapter afterward, promote /
deploy it via the dedicated path (serverless per-token serving of your own LoRA isn't
available yet).

In [11]:
final = training_client.save_weights_for_sampler("router-final").result()
print("final checkpoint:", getattr(final, "path", None))
service.close()
print("session closed")

final checkpoint: pyroworks/run-1f4951fc31b540fcaf5866b25d753aaa/router-final-1ca6ce72
session closed
